<a href="https://colab.research.google.com/github/srijan-ray/cs4650-final-project/blob/main/NLP_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Setup

In [1]:
!pip install -q accelerate peft bitsandbytes transformers trl==0.19.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00


In [2]:
import re, torch, os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import classification_report

In [3]:
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#### Access

In [5]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

SecretNotFoundError: Secret HF_TOKEN does not exist.

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

#### Global Vars

In [ ]:
SAMPLE_SIZE = 20 # change to None for full dataset

CLAUDE_MODEL_ID = "claude-sonnet-4-20250514"
ADAPTER_DIR = "/content/llama_lora_adapter"
USE_LORA = False

#### Models

In [ ]:
LLAMA_MODEL_ID = "NousResearch/Llama-2-7b-chat-hf" # "meta-llama/Llama-3.1-8B-Instruct"
BIOMISTRAL_MODEL_ID = "BioMistral/BioMistral-7B-DARE"
GEMMA_MODEL_ID = "google/gemma-7b"

In [ ]:
def make_config():
    USE_4BIT = USE_LORA
    COMPUTE_DTYPE = "float16"
    QUANTIZATION_TYPE = "nf4"
    USE_NESTED_QUANTIZATION = False

    bnb = BitsAndBytesConfig(
        load_in_4bit=USE_4BIT,
        bnb_4bit_quant_type=QUANTIZATION_TYPE,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=USE_NESTED_QUANTIZATION,
    )
    return bnb

In [ ]:
def make_model(model_name, bnb):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb,
        device_map={"": 0}
    )
    model.config.use_cache = False
    model.config.pretraining_tp = 1
    return model

In [ ]:
def make_tokenizer(model_name):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    return tokenizer

In [ ]:
def make_whole(model_name):
    bnb = make_config()
    model = make_model(model_name, bnb)
    tokenizer = make_tokenizer(model_name)
    return model, tokenizer

Change models name below to swap (GPU doesn't have enough memory to hold all three at the same time)

In [ ]:
model, tokenizer = make_whole(GEMMA_MODEL_ID) # CHANGE MODELS HERE

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

#### LoRA Configuration

In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model

In [ ]:
if USE_LORA:
    LORA_DROPOUT = 0.05
    LORA_ALPHA = 16
    LORA_R = 16
    TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

In [ ]:
if USE_LORA:
    peft_config = LoraConfig(
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        r=LORA_R,
        target_modules=TARGET_MODULES,
        task_type="CAUSAL_LM",
        bias="none"
    )

In [ ]:
if USE_LORA:
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

#### Dataset

In [ ]:
pubmed_ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled")
print(f"  Splits available: {list(pubmed_ds.keys())}")

split = pubmed_ds['train'].train_test_split(test_size=0.2, seed=42)
pubmed_train = split['train']
pubmed_eval = split['test']

if SAMPLE_SIZE:
    pubmed_train = pubmed_train.select(range(min(SAMPLE_SIZE, len(pubmed_train))))
    pubmed_eval = pubmed_eval.select(range(min(SAMPLE_SIZE, len(pubmed_eval))))

In [ ]:
# Format for LoRA
def format_example(example):
    return {
        "text": f"Question: {example['question']}\nContext: {' '.join(example['context']['contexts'])}\nAnswer: {example['long_answer']}"
    }
pubmed_train = pubmed_ds["train"].map(format_example)

In [ ]:
# Alternative dataset for training
medqa_ds = load_dataset("GBaker/MedQA-USMLE-4-options")
print(f"Splits available: {list(medqa_ds.keys())}")

medqa_test = medqa_ds.get('test', medqa_ds.get('validation', medqa_ds['train']))

if SAMPLE_SIZE:
    medqa_test = medqa_test.select(range(min(SAMPLE_SIZE, len(medqa_test))))

#### Fine Tuning

In [ ]:
if USE_LORA:
    from trl import SFTConfig, SFTTrainer
    from trl.trainer import ConstantLengthDataset

In [ ]:
if USE_LORA:
    # Number of training epochs
    num_train_epochs = 1

    ### BEGIN YOUR CODE ###

    # Select hyperparameters for learning rate
    optimizer = "paged_adamw_32bit"
    max_grad_norm = 0.3
    learning_rate = 0.0002
    weight_decay = 0.001

    ### END YOUR CODE ###

    # Select hyperparameters for learning rate scheduler
    lr_scheduler_type = "cosine"          # Learning rate schedule type
    warmup_ratio = 0.03                   # Ratio of steps for a linear warmup (from 0 to learning rate)

    # Etc. training configurations (ajudst for your compute requirements accordingly)
    fp16 = False                          # Enable fp16/bf16 training
    bf16 = False
    per_device_train_batch_size = 4       # Batch size per GPU for training
    gradient_accumulation_steps = 1       # Number of update steps to accumulate the gradients for
    gradient_checkpointing = True         # Enable gradient checkpointing
    save_steps = 0                        # Save checkpoint every X updates steps
    logging_steps = 25                    # Log every X updates steps

    # Options for supervised fine-tuning with TRL
    max_seq_length = 128
    group_by_length = True                # Group sequences into batches with same length
    packing = False                       # Pack multiple short examples in the same input sequence to increase efficiency

    training_arguments = SFTConfig(
        output_dir='.',
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        optim=optimizer,
        save_steps=save_steps,
        logging_steps=logging_steps,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        fp16=fp16,
        bf16=bf16,
        max_grad_norm=max_grad_norm,
        max_steps=-1,
        warmup_ratio=warmup_ratio,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        group_by_length=group_by_length,
        packing=packing,
        lr_scheduler_type=lr_scheduler_type,
        report_to="none"
    )

In [ ]:
if USE_LORA:
    trainer = SFTTrainer(
        model=model,
        train_dataset=pubmed_train, # set dataset here
        args=training_arguments,
        processing_class=tokenizer
    )

    trainer.train()

#### Evaluation Setup

In [ ]:
!pip install groq pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 11.5 MB/s eta 0:00:00


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from groq import Groq
import pandas as pd
import numpy as np
import time

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def zero_shot(model, tokenizer, dataset, idx):
    row = dataset[idx]
    prompt = f"Question: {row['question']}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):]

In [ ]:
def few_shot(model, tokenizer, dataset, idx, n_examples=3):
    examples = ""
    for i in range(n_examples):
        ex = dataset[i]
        examples += f"Question: {ex['question']}\nAnswer: {ex['long_answer']}\n\n"
    row = dataset[idx]
    prompt = f"{examples}Question: {row['question']}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):]

In [ ]:
def chain_of_thought(model, tokenizer, dataset, idx):
    row = dataset[idx]
    prompt = (
        f"Question: {row['question']}\n"
        "Let's think step by step before giving a final answer.\n"
        "Reasoning:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=300)
    return tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):]

In [ ]:
def eval_cosine_similarity(dataset, idx, model_response):
    reference = dataset[idx]["long_answer"]
    emb = embedder.encode([model_response, reference])
    return round(float(cosine_similarity([emb[0]], [emb[1]])[0][0]), 2)

In [ ]:
import re

def _parse_float(text):
    match = re.search(r"\b([0-1]?\.\d+|1\.0|0)\b", text.strip())
    return float(match.group(1)) if match else None

In [ ]:
def eval_grok_similarity(dataset, idx, model_response):
    reference = dataset[idx]["long_answer"]
    prompt = (
        f"Rate the semantic similarity between Response A and Response B on a scale from 0.0 to 1.0.\n"
        f"Response A: {model_response}\n"
        f"Response B: {reference}\n"
        "Do not include any text in your response, only return a float from 0.0 to 1.0."
    )
    resp = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
    )
    return _parse_float(resp.choices[0].message.content)

In [ ]:
def eval_grok_coherence(dataset, idx, model_response):
    reference = dataset[idx]["long_answer"]
    prompt = (
        f"Given this reference answer: {reference}\n"
        f"Rate the coherence of this response on a scale from 0.0 to 1.0: {model_response}\n"
        "Coherence means the response is logically consistent and factually aligned with the reference.\n"
        "Do not include any text in your response, only return a float from 0.0 to 1.0."
    )
    resp = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
    )
    return _parse_float(resp.choices[0].message.content)

#### Evaluation

In [ ]:
idx = 5 # Any sample id from 0 to SAMPLE_SIZE minus 1
zs  = zero_shot(model, tokenizer, pubmed_eval, idx)
fs  = few_shot(model, tokenizer, pubmed_eval, idx, n_examples=3)
cot = chain_of_thought(model, tokenizer, pubmed_eval, idx)

In [ ]:
print(eval_cosine_similarity(pubmed_eval, idx, zs))
print(eval_cosine_similarity(pubmed_eval, idx, fs))
print(eval_cosine_similarity(pubmed_eval, idx, cot))

print(eval_grok_similarity(pubmed_eval, idx, zs))
print(eval_grok_similarity(pubmed_eval, idx, fs))
print(eval_grok_similarity(pubmed_eval, idx, cot))

print(eval_grok_coherence(pubmed_eval, idx, zs))
print(eval_grok_coherence(pubmed_eval, idx, fs))
print(eval_grok_coherence(pubmed_eval, idx, cot))

0.02
0.42
0.24
0.0
0.24
0.23
0.9
0.5
0.8


In [ ]:
def build_eval_dataframe(model, tokenizer, dataset):

    records = []
    for idx in range(SAMPLE_SIZE):
        print(f"Processing {idx + 1}/{SAMPLE_SIZE}")
        row = dataset[idx]

        zs  = zero_shot(model, tokenizer, dataset, idx)
        fs  = few_shot(model, tokenizer, dataset, idx)
        cot = chain_of_thought(model, tokenizer, dataset, idx)

        records.append({
            "question":        row["question"],
            "long_answer":     row["long_answer"],
            "final_decision":  row["final_decision"],

            "zs_response":     zs,
            "fs_response":     fs,
            "cot_response":    cot,

            "zs_cosine":       eval_cosine_similarity(dataset, idx, zs),
            "fs_cosine":       eval_cosine_similarity(dataset, idx, fs),
            "cot_cosine":      eval_cosine_similarity(dataset, idx, cot),

            "zs_grok_sim":     eval_grok_similarity(dataset, idx, zs),
            "fs_grok_sim":     eval_grok_similarity(dataset, idx, fs),
            "cot_grok_sim":    eval_grok_similarity(dataset, idx, cot),

            "zs_coherence":    eval_grok_coherence(dataset, idx, zs),
            "fs_coherence":    eval_grok_coherence(dataset, idx, fs),
            "cot_coherence":   eval_grok_coherence(dataset, idx, cot),
        })

    return pd.DataFrame(records)

In [ ]:
df.to_csv("model_responses.csv", index=False)
print(f"Exported {len(df)} rows to model_responses.csv")

Exported 20 rows to model_responses.csv


#### Sample Data For Submission

In [ ]:
# Export a data sample to .csv
sample = pubmed_train.select(range(min(100, SAMPLE_SIZE)))
df = pd.DataFrame(sample).drop(columns=["text"])
df.to_csv("pubmed_sample.csv", index=False)
print(f"Exported {len(df)} rows to pubmed_sample.csv")

Exported 20 rows to pubmed_sample.csv
